# Q1 + readout resonator: the full loop

1. **LOM first:** turn target f_q, α, f_r and g into Lj, C_Σ, Cg and the λ/4 length.
2. **Layout:** edit `design.json` and build. KLayout reloads live through klive. Start KLayout with klive enabled first.
3. **Simulate:** export a HFSS/Q3D job for the `Q1R1` group, run it in AEDT, then parse the results.
4. **Extract:** run Metal LOM, and cross-check with scqubits, QuCat and QuTiP.
5. **Feedback:** tune geometry knobs toward the targets.

**`KLAYOUT_GUI`** (next to `LIVE`): `True` opens the KLayout window when it is not running (≈ 15–20 s the first time, with the layout and the Quantcore plugin), reopens it if you closed it, and brings it to the front on the first build; `False` only redraws a KLayout you already have open (klive).

In [ ]:
import json, os, pathlib
import quantcore_circuits as qcc
from quantcore_circuits.analysis.synthesis import synthesize_transmon_readout

LIVE = os.environ.get("QCC_LIVE", "1") == "1"   # live KLayout by default; QCC_LIVE=0 for batch runs
KLAYOUT_GUI = True                               # True: open / raise the KLayout window; False: redraw an open one only
from quantcore_circuits.live.klayout import use_gui
use_gui(KLAYOUT_GUI)
from quantcore_circuits import track
TRACKING = track.start("q1_r1_loop")            # dashboard: records into build/q1_r1_loop (scripts\dashboard.cmd)
DESIGN = pathlib.Path("q1_r1_feedline.json")
targets = dict(f_q_GHz=5.1, alpha_MHz=-220, f_r_GHz=7.2, g_MHz=90)
synth = synthesize_transmon_readout(targets["f_q_GHz"], targets["alpha_MHz"], targets["f_r_GHz"], targets["g_MHz"])
{k: round(v, 3) for k, v in synth.items() if isinstance(v, float)}

### The FEM part in two stages, and its switches

As in the tutorials: (1) a **subset check**, FEM only (the Q3D of step 3: did it converge, are the values sane?),
then (2) the extraction and feedback of steps 4–5. **`SIM_SUBSET`** says which part of the chip goes into the Q3D job of
step 3: `[("Q1", "R1")]` (the design's sim group `Q1R1`: the qubit and its λ/4, the feedline cut by the box; this chip's
only pair) or, with `FEM_WHOLE = True` (opt-in), `"whole"` (the whole chip with its pads, the built-in group `chip`). The optimiser of step 5 always
re-solves `Q1R1`. **`RUN_FEM = True`** starts AEDT (step 3's solve, every optimiser step); `False` starts no AEDT and
uses the kept / earlier results. `FEM_FAST` and `AEDT_MODE` sit here too.

In [ ]:
SIM_SUBSET = [("Q1", "R1")]      # the pair to render and simulate, or "whole" (the chip with its pads)
FEM_WHOLE = False                # True (opt-in, slower): the whole chip with its pads instead of the subset
if FEM_WHOLE:
    SIM_SUBSET = "whole"
RUN_FEM = False                  # True: solve in AEDT (step 3 and the optimiser of step 5); False: no AEDT
FEM_FAST = True                  # True: fast test settings (Q3D 2 %, ≤ 8 passes), optimiser jobs too
AEDT_MODE = "batch"              # "batch" (no window) | "gui" (a new AEDT window) | "attach" (yours)
SIM_GROUP = "chip" if SIM_SUBSET == "whole" else "Q1R1"   # this chip has one pair: its group is Q1R1

## 2. Layout: edit, build and view in KLayout

In [ ]:
design = qcc.load_design(DESIGN)
data = design.to_json_dict()
data["instances"]["Q1"]["junction"]["Lj_nH"] = round(synth["Lj_nH"], 3)       # from LOM synthesis
data["routes"]["R1"]["length"] = round(synth["resonator_length_um"], 1)        # λ/4 at f_r
design = qcc.load_design(data)

if LIVE:
    result = qcc.show(design, "build/q1_r1_feedline.oas")   # build + write + klive reload
else:
    result = qcc.build(design)
    qcc.write_layout(result, "build/q1_r1_feedline.oas")
result.route_lengths()

## 3. Simulate (the subset check): HFSS/Q3D C-matrix for the Q1R1 group

**`AEDT_MODE`**: `"batch"` solves in a new non-graphical AEDT; `"gui"` opens a new AEDT window and leaves it open, so you watch the mesh, the convergence and the fields; `"attach"` runs inside the AEDT window you already have open (or opens one) and leaves the project there. Every mode prints a progress line (step, pass, mesh, ΔF / ΔS / ΔC) every 30 s, so a stuck run is visible; on a timeout only what qcc started is stopped.

In [ ]:
from quantcore_circuits.sim_tools import build_simulation, export_job
from quantcore_circuits.sim_tools.runner import run_job

from quantcore_circuits.sim_tools.export import set_fem_fast
set_fem_fast(FEM_FAST)
setup = build_simulation(design, SIM_GROUP, "q3d_cmatrix")   # SIM_SUBSET
job = export_job(setup, "build/jobs/q1r1_q3d" if SIM_GROUP == "Q1R1" else f"build/jobs/q1r1_{SIM_GROUP}_q3d",
                 overwrite=True)
[(p.number, p.kind, p.name) for p in setup.ports]

In [ ]:
# Runs AEDT (qcc_solve.py) on this machine the AEDT_MODE way; this can take several minutes.
# AEDT_MODE: the SIM_SUBSET cell
from quantcore_circuits.sim_tools.ansys import find_ansys_executable
HAVE_AEDT = find_ansys_executable() is not None   # AEDT 2024 R2 (or QCC_ANSYS_EXECUTABLE)
status = (run_job(job, aedt=AEDT_MODE) if RUN_FEM and HAVE_AEDT else
          {"status": "skipped: " + ("RUN_FEM = False" if HAVE_AEDT else "AEDT 2024 R2 not installed")})
status["status"]

## 4. Extract parameters

In [ ]:
from quantcore_circuits.analysis.results import collect_results
from quantcore_circuits.analysis.extract import extract, compare
from quantcore_circuits.analysis.store import ResultsStore

res = collect_results(job)
if "cmatrix" not in res:   # not solved here: use the Q1R1 Q3D job solved earlier for this design
    res = collect_results("../build/jobs/q1_r1_feedline_Q1R1_q3d_cmatrix")
params = extract(design, res)
ResultsStore("results.sqlite").put(res["design_hash"], "Q1R1", "q3d_cmatrix", params, design=design.name, job_dir=job)
compare(design.targets, params)

In [ ]:
# Cross-checks from the same C-matrix
import numpy as np
from quantcore_circuits.analysis.circuits import qucat_from_cmatrix, qutip_transmon_resonator
C = np.array(res["cmatrix"]["matrix"]); labels = res["cmatrix"]["labels"]
q = qucat_from_cmatrix(C, labels, "Q1.island", design.instances["Q1"].junction.Lj_nH,
                       resonators={"Q1.claw_n": (params["R1"]["f_GHz"], 50.0)})
jc = qutip_transmon_resonator(params["Q1"]["f_q_GHz"], params["Q1"]["alpha_MHz"], params["R1"]["f_GHz"], params["R1"]["g_MHz"])
q["f_q_GHz"], q["alpha_MHz"], jc["chi_MHz"]

## 5. Feedback: tune the cross length to hit f_q

`pipeline_evaluator` runs build → job → AEDT → extract for every iteration. Each step is a full simulation.

In [ ]:
from quantcore_circuits.optimise import optimise, pipeline_evaluator

out = None
if RUN_FEM and HAVE_AEDT:                        # every iteration is a full AEDT solve
    evaluate = pipeline_evaluator("Q1R1", "q3d_cmatrix")
    # ≤ 410 µm: a longer cross moves the flux port past the FLUX route's fixed waypoint (y = 1967 µm) and the route
    # folds back (RoutingError); at the bound the optimiser stops and reports it
    out = optimise(design, {"Q1.cross_length": (200, 410)}, {"Q1.f_q_GHz": targets["f_q_GHz"]}, evaluate,
                   tolerance={"Q1.f_q_GHz": 0.05}, max_iter=6)
    print(out.converged, [(it.knobs, it.error) for it in out.history])
else:
    print("skipped: the optimiser solves in AEDT at every iteration (RUN_FEM = True"
          + ("" if HAVE_AEDT else ", and AEDT 2024 R2 installed") + ")")

In [ ]:
if out is not None:
    qcc.save_design(out.design, "q1_r1_feedline_tuned.json")
    if LIVE:
        qcc.show(out.design)